# AWM and GFAR: recommendation experiments

A reproducible comparison on TravelSync's official v2 POIs using controlled synthetic groups.

**Read:** [Findings](#findings) | [Baseline](#baseline) | [GFAR depth](#depth) | [AWM threshold](#threshold) | [Candidate pools](#pools) | [Equal length](#equal) | [Limitations](#limitations)

**Run:** select the project's `.venv` kernel, then **Restart Kernel -> Run All**. Install with `uv sync --extra ingestion` (pandas supplies the tables). The notebook works from the repository root or `notebooks/`. Exports always go to the repository's ignored `data/generated/evaluation/compare_recommenders/` directory.

In [106]:
import hashlib
import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from functools import lru_cache
from importlib.resources import files
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from ourney_travelsync import AverageWithoutMisery, FEATURE_NAMES, GFAR, load_places
from ourney_travelsync.evaluation import evaluate_satisfaction
from ourney_travelsync.places import OFFICIAL_FILENAME, OFFICIAL_SHA256
from ourney_travelsync.recommenders import calculate_preference_score, prepare_group

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file() and (p / 'src/ourney_travelsync').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Run from the repository root or its notebooks directory.')
OUTPUT = ROOT / 'data/generated/evaluation/compare_recommenders'
SEEDS = tuple(range(10))
CANDIDATE_TOP_N, TOP_K = 100, 10
AWM_THRESHOLD, GFAR_DEPTH = 2.5, 20
DEPTHS = (10, 20, 50, 100)
THRESHOLDS = (1.0, 1.5, 2.0, 2.5, 3.0)
resource = files('ourney_travelsync').joinpath('data', OFFICIAL_FILENAME)
dataset_hash = hashlib.sha256(resource.read_bytes()).hexdigest()
assert dataset_hash == OFFICIAL_SHA256, 'Dataset differs from the official v2 snapshot.'
places = load_places()
metadata = {r['place_id']: r for r in places.records}


def table(frame, columns=None):
    view = frame.loc[:, columns] if columns is not None else frame
    labels = {
        'scenario': 'Scenario', 'algorithm': 'Algorithm', 'gfar_depth': 'Depth',
        'awm_threshold': 'Threshold', 'pool': 'Pool', 'candidates_min': 'Candidates min',
        'candidates_max': 'Candidates max', 'returned_min': 'Returned min',
        'returned_max': 'Returned max', 'empty_runs': 'Empty runs',
        'mean_utility': 'Mean utility', 'minimum_member_utility': 'Minimum member',
        'minimum_member_low': 'Minimum member low', 'minimum_member_high': 'Minimum member high',
        'minority_utility': 'Minority utility', 'disparity': 'Disparity',
        **{f'user_{i}_utility': f'User {i}' for i in range(1, 5)},
    }
    display(view.rename(columns=labels).style.format(precision=3, na_rep='N/A').hide(axis='index'))


print(f'Python {platform.python_version()} | {sys.executable}')
print(f'{OFFICIAL_FILENAME}: {len(places.place_ids):,} POIs | SHA256 verified')
print(f'Output: {OUTPUT}')

Python 3.13.15 | d:\Backup\Documents\Inha '23\8th semester\Capstone Design in CSE\TravelSync\Ourney_TravelSync\.venv\Scripts\python.exe
travelsync_final_places_v2.csv: 3,570 POIs | SHA256 verified
Output: d:\Backup\Documents\Inha '23\8th semester\Capstone Design in CSE\TravelSync\Ourney_TravelSync\data\generated\evaluation\compare_recommenders


## Method and measures

Within each comparison, AWM and GFAR receive the **same scored pool, group, seed, and requested list length**. Each individual's score is their mean preference across a POI's active binary features, on the original **1-5 scale**.

**AWM** rejects a POI if any member's score falls below its threshold, then ranks survivors by group mean. **GFAR** greedily selects using normalized Borda relevance from each member's top-depth ranking. GFAR selection gains and AWM output scores have different meanings; evaluation always uses the original individual scores. This repository's GFAR implementation still requires validation against the authors' implementation.

**Candidate top-N** determines the shared candidate union. **GFAR depth** determines relevance inside that pool. These are separate parameters.

| Measure | Definition | Reading |
|---|---|---|
| Mean utility | Mean of members' average scores across the returned list | Higher predicted group preference |
| Minimum member utility | Lowest member average across the list | Higher predicted utility for the least-satisfied member |
| Disparity | Population standard deviation of member list averages | Lower member-level spread |
| Returned min / max | Smallest / largest returned count across seeds | Compare with requested K |
| Empty runs | Runs returning no recommendations | Utility is undefined: N/A, never zero |

Tables average per-run metrics over **seeds 0-9**. Utility averages exclude empty runs; returned ranges and empty counts include every run. Seed ranges measure tie sensitivity, not confidence intervals: seeds perturb candidate identities/order among ties, not user profiles. A minimum member average does not guarantee every returned POI is acceptable to that member.

## Synthetic groups

Two deliberately contrasting profiles form three four-member groups. Identical profiles isolate preference conflict; these fixtures are not a sample of real users. In the minority scenario, **user_4** is the nature-oriented member.

In [107]:
def profile(**preferences):
    return dict.fromkeys(FEATURE_NAMES, 1) | preferences


food_profile = profile(food=5, cafe=5, shopping=4, entertainment=4)
nature_profile = profile(nature=5, history=5, activity=4, relax=4)
scenarios = {
    'aligned': [food_profile] * 4,
    'conflicting': [food_profile] * 2 + [nature_profile] * 2,
    'minority': [food_profile] * 3 + [nature_profile],
}
table(pd.DataFrame({
    'Feature': FEATURE_NAMES,
    'Food-oriented': [food_profile[f] for f in FEATURE_NAMES],
    'Nature-oriented': [nature_profile[f] for f in FEATURE_NAMES],
}))
table(pd.DataFrame([
    {'Scenario': name, 'Food members': food, 'Nature members': nature}
    for name, food, nature in [('aligned', 4, 0), ('conflicting', 2, 2), ('minority', 3, 1)]
]))

Feature,Food-oriented,Nature-oriented
food,5,1
cafe,5,1
nature,1,5
history,1,5
art,1,1
shopping,4,1
activity,1,4
entertainment,4,1
relax,1,4
festival,1,1


Scenario,Food members,Nature members
aligned,4,0
conflicting,2,2
minority,3,1


## Experiment runner

One cached candidate group is reused by every setting with the same scenario, pool, and seed. The runner records every member's list utility and every selected POI's identity and individual scores. It checks AWM's per-POI acceptance rule. Re-running the execution cell replaces its previous results.

In [108]:
@lru_cache(maxsize=None)
def candidate_group(scenario, pool, seed):
    if pool not in ('top100', 'full'):
        raise ValueError(f'Unknown pool: {pool}')
    members = {f'user_{i+1}': dict(p) for i, p in enumerate(scenarios[scenario])}
    limit = CANDIDATE_TOP_N if pool == 'top100' else len(places.place_ids)
    group = prepare_group(places, members, individual_top_n=limit, tie_seed=seed)
    group.validate(require_scores=True)
    return group


runs, selected_pois = [], []


def run_experiment(experiment, scenario, algorithm, parameter, seed, pool='top100', top_k=TOP_K):
    group = candidate_group(scenario, pool, seed)
    model = AverageWithoutMisery(parameter) if algorithm == 'AWM' else GFAR(individual_top_n=int(parameter))
    recommendations = model.recommend(group, top_k=top_k)
    metrics = evaluate_satisfaction(group, recommendations)
    member_values = list(metrics['member_satisfaction'].values())
    indices = {poi_id: i for i, poi_id in enumerate(group.poi_ids)}
    run_id = f'{experiment}/{scenario}/{pool}/{algorithm}/{parameter}/k{top_k}/s{seed}'
    row = {
        'run_id': run_id, 'experiment': experiment, 'scenario': scenario, 'pool': pool,
        'candidate_top_n': CANDIDATE_TOP_N if pool == 'top100' else len(places.place_ids),
        'candidate_count': len(group.poi_ids), 'seed': seed, 'algorithm': algorithm,
        'awm_threshold': parameter if algorithm == 'AWM' else None,
        'gfar_depth': int(parameter) if algorithm == 'GFAR' else None,
        'requested_pois': top_k, 'returned_pois': metrics['returned_pois'],
        'mean_utility': metrics['mean_satisfaction'],
        'minimum_member_utility': metrics['minimum_member_satisfaction'],
        'disparity': float(np.std(member_values)) if recommendations else None,
        'minority_utility': metrics['member_satisfaction']['user_4'] if scenario == 'minority' else None,
        **{f'{u}_utility': v for u, v in metrics['member_satisfaction'].items()},
    }
    runs.append(row)
    for rank, result in enumerate(recommendations, 1):
        scores = group.individual_scores[:, indices[result.poi_id]]
        if algorithm == 'AWM':
            assert np.all(scores >= parameter), 'AWM acceptance rule violated.'
        selected_pois.append({
            'run_id': run_id, 'rank': rank, 'poi_id': result.poi_id,
            'place_name': metadata[result.poi_id]['place_name'], 'algorithm_score': float(result.score),
            **{f'{u}_utility': float(v) for u, v in zip(group.user_ids, scores)},
        })


def summarize(experiment, by):
    frame = results_frame.loc[results_frame['experiment'].eq(experiment)]
    return frame.groupby(by, sort=False, dropna=False).agg(
        runs=('seed', 'size'),
        candidates_min=('candidate_count', 'min'), candidates_max=('candidate_count', 'max'),
        returned_min=('returned_pois', 'min'), returned_max=('returned_pois', 'max'),
        empty_runs=('returned_pois', lambda c: int(c.eq(0).sum())),
        valid_runs=('mean_utility', 'count'), mean_utility=('mean_utility', 'mean'),
        minimum_member_utility=('minimum_member_utility', 'mean'),
        minimum_member_low=('minimum_member_utility', 'min'),
        minimum_member_high=('minimum_member_utility', 'max'),
        disparity=('disparity', 'mean'), minority_utility=('minority_utility', 'mean'),
    ).reset_index()


METRICS = ['returned_min', 'returned_max', 'empty_runs', 'mean_utility', 'minimum_member_utility', 'disparity']

In [109]:
runs.clear()
selected_pois.clear()
for scenario in scenarios:
    for seed in SEEDS:
        for algorithm, parameter in [('AWM', AWM_THRESHOLD), ('GFAR', GFAR_DEPTH)]:
            run_experiment('baseline', scenario, algorithm, parameter, seed)
        for depth in DEPTHS:
            run_experiment('depth', scenario, 'GFAR', depth, seed)
        for threshold in THRESHOLDS:
            run_experiment('threshold', scenario, 'AWM', threshold, seed)
        if scenario != 'aligned':
            for pool in ('top100', 'full'):
                for algorithm, parameter in [('AWM', AWM_THRESHOLD), ('GFAR', 10)]:
                    run_experiment('pool', scenario, algorithm, parameter, seed, pool=pool)
            for algorithm, parameter in [('AWM', AWM_THRESHOLD), ('GFAR', 10)]:
                run_experiment('equal_length', scenario, algorithm, parameter, seed, pool='full', top_k=5)

results_frame = pd.DataFrame(runs)
selected_frame = pd.DataFrame(selected_pois)
assert len(results_frame) == 450
assert results_frame['run_id'].is_unique
assert results_frame['returned_pois'].le(results_frame['requested_pois']).all()
assert results_frame.loc[results_frame['returned_pois'].eq(0), 'mean_utility'].isna().all()
assert selected_frame.groupby('run_id').size().to_dict() == {
    r['run_id']: r['returned_pois'] for r in runs if r['returned_pois']
}
baseline = summarize('baseline', ['scenario', 'algorithm'])
depth_summary = summarize('depth', ['scenario', 'gfar_depth'])
threshold_summary = summarize('threshold', ['scenario', 'awm_threshold'])
pool_summary = summarize('pool', ['scenario', 'pool', 'algorithm'])
equal_summary = summarize('equal_length', ['scenario', 'algorithm'])
print(f'Completed {len(results_frame)} runs across five experiments; {len(selected_frame):,} selected-POI records.')

Completed 450 runs across five experiments; 3,000 selected-POI records.


<a id="findings"></a>
## Results at a glance

Findings and tables below are generated from this execution. The notebook does not change production defaults.

In [110]:
def count_range(low, high):
    return str(int(low)) if low == high else f'{int(low)} to {int(high)}'


minority_depth = depth_summary.loc[depth_summary['scenario'].eq('minority')].set_index('gfar_depth')
minority_awm = pool_summary.loc[pool_summary['scenario'].eq('minority') & pool_summary['algorithm'].eq('AWM')].set_index('pool')
display(Markdown(
    f"- **Candidate generation changes the AWM outcome.** At threshold {AWM_THRESHOLD}, the minority fixture returns "
    f"{count_range(minority_awm.loc['top100', 'returned_min'], minority_awm.loc['top100', 'returned_max'])} POIs from the top-100 union, "
    f"versus {count_range(minority_awm.loc['full', 'returned_min'], minority_awm.loc['full', 'returned_max'])} from the full pool (requested: {TOP_K}).\n"
    f"- **GFAR depth changes minority utility.** It is {minority_depth.loc[10, 'minority_utility']:.3f} at depth 10 "
    f"and {minority_depth.loc[100, 'minority_utility']:.3f} at depth 100. A higher group mean can conceal a worse member outcome.\n"
    "- **Compare equal-length lists.** The table below requests five POIs from the full pool for both algorithms. "
    "Read mean and minimum member utility together."
))
table(equal_summary, ['scenario', 'algorithm', 'returned_min', 'returned_max', 'mean_utility', 'minimum_member_utility'])

- **Candidate generation changes the AWM outcome.** At threshold 2.5, the minority fixture returns 0 POIs from the top-100 union, versus 5 from the full pool (requested: 10).
- **GFAR depth changes minority utility.** It is 2.200 at depth 10 and 1.000 at depth 100. A higher group mean can conceal a worse member outcome.
- **Compare equal-length lists.** The table below requests five POIs from the full pool for both algorithms. Read mean and minimum member utility together.

Scenario,Algorithm,Returned min,Returned max,Mean utility,Minimum member
conflicting,AWM,5,5,2.500,2.500
conflicting,GFAR,5,5,3.000,2.600
minority,AWM,5,5,2.500,2.500
minority,GFAR,5,5,3.600,1.800


<a id="baseline"></a>
## 1. Baseline: shared top-100 union

**K = 10 | AWM threshold = 2.5 | GFAR depth = 20 | ten seeds.** Threshold 1.5 belongs to the sensitivity sweep, not this baseline.

An empty AWM list means the candidate pool has no qualifying POIs; it does not establish dataset-wide incompatibility. A complete GFAR list does not guarantee each POI is acceptable to every member.

In [111]:
table(baseline, ['scenario', 'algorithm', *METRICS])
member_columns = [f'user_{i}_utility' for i in range(1, 5)]
minority_members = results_frame.loc[
    results_frame['experiment'].eq('baseline') & results_frame['scenario'].eq('minority')
].groupby('algorithm', sort=False)[member_columns].mean().reset_index()
display(Markdown('**Minority group: each member\'s mean list utility.** Users 1-3 have the food profile; user 4 has the nature profile.'))
table(minority_members)

Scenario,Algorithm,Returned min,Returned max,Empty runs,Mean utility,Minimum member,Disparity
aligned,AWM,10,10,0,5.000,5.000,0.000
aligned,GFAR,10,10,0,5.000,5.000,0.000
conflicting,AWM,0,0,10,N/A,N/A,N/A
conflicting,GFAR,10,10,0,3.000,3.000,0.000
minority,AWM,0,0,10,N/A,N/A,N/A
minority,GFAR,10,10,0,3.800,1.400,1.386


**Minority group: each member's mean list utility.** Users 1-3 have the food profile; user 4 has the nature profile.

Algorithm,User 1,User 2,User 3,User 4
AWM,N/A,N/A,N/A,N/A
GFAR,4.600,4.600,4.600,1.400


<a id="depth"></a>
## 2. GFAR ranking-depth sensitivity

**Top-100 union | K = 10 | depths 10, 20, 50, 100 | ten seeds.** Only relevance depth changes; candidate generation stays fixed.

The first table highlights the minority tradeoff. Low/high columns give the observed per-run extremes of minimum member utility across seeds. All scenarios are retained below and in the exported summary.

In [112]:
table(depth_summary.loc[depth_summary['scenario'].eq('minority')],
      ['gfar_depth', 'returned_min', 'returned_max', 'mean_utility', 'minority_utility',
       'minimum_member_low', 'minimum_member_high', 'disparity'])
table(depth_summary.loc[~depth_summary['scenario'].eq('minority')],
      ['scenario', 'gfar_depth', 'mean_utility', 'minimum_member_utility'])

Depth,Returned min,Returned max,Mean utility,Minority utility,Minimum member low,Minimum member high,Disparity
10.000,10,10,3.400,2.200,2.200,2.200,0.693
20.000,10,10,3.800,1.400,1.400,1.400,1.386
50.000,10,10,4.000,1.000,1.000,1.000,1.732
100.000,10,10,4.000,1.000,1.000,1.000,1.732


Scenario,Depth,Mean utility,Minimum member
aligned,10.000,5.000,5.000
aligned,20.000,5.000,5.000
aligned,50.000,5.000,5.000
aligned,100.000,5.000,5.000
conflicting,10.000,3.000,3.000
conflicting,20.000,3.000,3.000
conflicting,50.000,3.000,3.000
conflicting,100.000,3.000,3.000


<a id="threshold"></a>
## 3. AWM threshold sensitivity

**Top-100 union | K = 10 | thresholds 1.0, 1.5, 2.0, 2.5, 3.0 | ten seeds.** Only the acceptance threshold changes.

Threshold 1.0 removes the veto on this score scale, giving mean-score ranking. Higher thresholds can eliminate all candidates. These results apply to the truncated pool; full-pool recovery below is tested specifically at 2.5.

In [113]:
table(threshold_summary, ['scenario', 'awm_threshold', 'returned_min', 'returned_max',
                          'empty_runs', 'mean_utility', 'minimum_member_utility'])

Scenario,Threshold,Returned min,Returned max,Empty runs,Mean utility,Minimum member
aligned,1.000,10,10,0,5.000,5.000
aligned,1.500,10,10,0,5.000,5.000
aligned,2.000,10,10,0,5.000,5.000
aligned,2.500,10,10,0,5.000,5.000
aligned,3.000,10,10,0,5.000,5.000
conflicting,1.000,10,10,0,3.000,1.720
conflicting,1.500,0,0,10,N/A,N/A
conflicting,2.000,0,0,10,N/A,N/A
conflicting,2.500,0,0,10,N/A,N/A
conflicting,3.000,0,0,10,N/A,N/A


<a id="pools"></a>
## 4. Candidate-pool expansion

**K = 10 | AWM threshold = 2.5 | GFAR depth = 10 | ten seeds.** Compare top-100 unions with all POIs. This extends the original seed-0 exploration to all ten seeds.

Compromise places can pass every member's acceptance threshold while falling outside all individual top-100 lists. GFAR depth remains 10 as the pool expands. Equal utility does not establish identical selected POI IDs.

In [114]:
table(pool_summary, ['scenario', 'pool', 'algorithm', 'candidates_min', 'candidates_max',
                     'returned_min', 'returned_max', 'mean_utility', 'minimum_member_utility'])

Scenario,Pool,Algorithm,Candidates min,Candidates max,Returned min,Returned max,Mean utility,Minimum member
conflicting,top100,AWM,200,200,0,0,N/A,N/A
conflicting,top100,GFAR,200,200,10,10,3.000,3.000
conflicting,full,AWM,3570,3570,5,5,2.500,2.500
conflicting,full,GFAR,3570,3570,10,10,3.000,3.000
minority,top100,AWM,200,200,0,0,N/A,N/A
minority,top100,GFAR,200,200,10,10,3.400,2.200
minority,full,AWM,3570,3570,5,5,2.500,2.500
minority,full,GFAR,3570,3570,10,10,3.400,2.200


### Inspect the compromise places

The following table lists **every full-pool POI meeting AWM's 2.5 threshold**, using the minority fixture and seed 0. Profile scores explain acceptance; the last column checks retention in the top-100 union. The conflicting fixture uses the same two profiles, so its acceptance condition is identical.

In [115]:
full_group = candidate_group('minority', 'full', 0)
short_ids = set(candidate_group('minority', 'top100', 0).poi_ids)
acceptable = np.flatnonzero((full_group.individual_scores >= AWM_THRESHOLD).all(axis=0))
compromise_rows = []
for i in acceptable:
    poi_id = full_group.poi_ids[i]
    compromise_rows.append({
        'POI ID': poi_id, 'Place': metadata[poi_id]['place_name'],
        'Active features': ', '.join(f for f, active in zip(FEATURE_NAMES, full_group.poi_features[i]) if active),
        'Food profile score': float(full_group.individual_scores[0, i]),
        'Nature profile score': float(full_group.individual_scores[3, i]),
        'In top-100 union': poi_id in short_ids,
    })
compromises = pd.DataFrame(compromise_rows)
table(compromises)
print(f'{len(compromises)} qualifying full-pool POIs; {sum(r["In top-100 union"] for r in compromise_rows)} retained in the seed-0 top-100 union.')

POI ID,Place,Active features,Food profile score,Nature profile score,In top-100 union
kakao:53918924,부산광역시교육청유아교육진흥원 유아놀이체험센터,"activity, entertainment",2.500,2.500,False
tour:229912,씨라이프부산아쿠아리움,"activity, entertainment",2.500,2.500,False
kakao:1123259027,어린이VR재난안전체험교육장,"activity, entertainment",2.500,2.500,False
kakao:1810485368,가상현실스포츠체험센터,"activity, entertainment",2.500,2.500,False
tour:769761,다대포 꿈의 낙조분수,"entertainment, relax",2.500,2.500,False


5 qualifying full-pool POIs; 0 retained in the seed-0 top-100 union.


<a id="equal"></a>
## 5. Equal-length comparison

**Full pool | K = 5 for both algorithms | AWM threshold = 2.5 | GFAR depth = 10 | ten seeds.** K = 5 follows the exploratory discovery of five acceptable AWM places; it is not an independently chosen validation setting.

Mean utility and minimum member utility measure different goals. A higher group mean alone does not establish stronger minority protection. AWM's veto applies to every returned POI; GFAR has no equivalent acceptance guarantee.

In [116]:
table(equal_summary, ['scenario', 'algorithm', *METRICS, 'minimum_member_low', 'minimum_member_high'])
minority_equal = equal_summary.loc[equal_summary['scenario'].eq('minority')].set_index('algorithm')
display(Markdown(
    f"**Minority tradeoff at equal length:** GFAR's mean utility is {minority_equal.loc['GFAR', 'mean_utility']:.3f} "
    f"versus AWM's {minority_equal.loc['AWM', 'mean_utility']:.3f}, while its minimum member utility is "
    f"{minority_equal.loc['GFAR', 'minimum_member_utility']:.3f} versus AWM's {minority_equal.loc['AWM', 'minimum_member_utility']:.3f}. "
    "These fixtures support a tradeoff, not a universal winner."
))

Scenario,Algorithm,Returned min,Returned max,Empty runs,Mean utility,Minimum member,Disparity,Minimum member low,Minimum member high
conflicting,AWM,5,5,0,2.500,2.500,0.000,2.500,2.500
conflicting,GFAR,5,5,0,3.000,2.600,0.400,2.600,2.600
minority,AWM,5,5,0,2.500,2.500,0.000,2.500,2.500
minority,GFAR,5,5,0,3.600,1.800,1.039,1.800,1.800


**Minority tradeoff at equal length:** GFAR's mean utility is 3.600 versus AWM's 2.500, while its minimum member utility is 1.800 versus AWM's 2.500. These fixtures support a tradeoff, not a universal winner.

## Appendix: original two-member example

The Alice/Bob example is preserved separately from the synthetic experiments. Both algorithms use the same seed-0 top-100 pool and K = 10. The tables use shared individual utilities, not algorithm-native gains.

In [117]:
example_users = {
    'alice': dict(zip(FEATURE_NAMES, [4, 5, 3, 2, 2, 3, 3, 3, 4, 1])),
    'bob': dict(zip(FEATURE_NAMES, [3, 2, 5, 4, 3, 2, 4, 2, 5, 3])),
}
example_poi = dict.fromkeys(FEATURE_NAMES, 0) | {'food': 1, 'cafe': 1}
assert calculate_preference_score(example_users['alice'], example_poi) == 4.5
example_group = prepare_group(places, example_users, individual_top_n=CANDIDATE_TOP_N, tie_seed=0)
example_indices = {poi_id: i for i, poi_id in enumerate(example_group.poi_ids)}
example_metrics, example_recommendations = [], []
for name, model in [('AWM', AverageWithoutMisery(AWM_THRESHOLD)), ('GFAR', GFAR(GFAR_DEPTH))]:
    recommendations = model.recommend(example_group, top_k=TOP_K)
    metrics = evaluate_satisfaction(example_group, recommendations)
    example_metrics.append({
        'Algorithm': name, 'Returned': metrics['returned_pois'],
        'Mean utility': metrics['mean_satisfaction'],
        'Minimum member utility': metrics['minimum_member_satisfaction'], **metrics['member_satisfaction'],
    })
    for rank, result in enumerate(recommendations, 1):
        scores = example_group.individual_scores[:, example_indices[result.poi_id]]
        example_recommendations.append({
            'Algorithm': name, 'Rank': rank, 'POI ID': result.poi_id,
            'Place': metadata[result.poi_id]['place_name'],
            'Alice utility': float(scores[0]), 'Bob utility': float(scores[1]),
        })
table(pd.DataFrame(example_metrics))
table(pd.DataFrame(example_recommendations))

Algorithm,Returned,Mean utility,Minimum member utility,alice,bob
AWM,10,4.375,3.750,3.750,5.000
GFAR,10,3.875,3.500,4.250,3.500


Algorithm,Rank,POI ID,Place,Alice utility,Bob utility
AWM,1,tour:2875837,달맞이동산,4.000,5.000
AWM,2,kakao:779442429,베니키아호텔사우나,4.000,5.000
AWM,3,tour:127149,민주공원,4.000,5.000
AWM,4,kakao:643003723,대성관온천,4.000,5.000
AWM,5,kakao:8560034,태종대온천,4.000,5.000
AWM,6,kakao:184057678,세실공원,3.500,5.000
AWM,7,kakao:25038624,남항대교 수변공원,3.500,5.000
AWM,8,tour:2758469,소두방공원,3.500,5.000
AWM,9,kakao:25034208,거울바위문화생활쉼터,3.500,5.000
AWM,10,kakao:166662665,동대공원,3.500,5.000


## Evidence and reproducibility

`runs.csv` records all parameters, candidate/returned counts, and member utilities. `selected_pois.csv` joins through `run_id` and records ranked identities, original member scores, and algorithm-native scores for traceability. Summary CSVs contain every scenario. Blanks mean undefined utility or inapplicable parameters.

The manifest records execution time, dataset hash, package versions, Git revision/status, and source hashes. A dirty checkout is recorded because a commit alone cannot identify uncommitted code. Generated files are ignored by Git; archive them explicitly when sharing. Saved notebook outputs are freshly executed, replacing the earlier transcript-based report.

In [118]:
OUTPUT.mkdir(parents=True, exist_ok=True)
results_frame.to_csv(OUTPUT / 'runs.csv', index=False)
selected_frame.to_csv(OUTPUT / 'selected_pois.csv', index=False)
compromises.to_csv(OUTPUT / 'compromise_pois.csv', index=False)
pd.DataFrame(example_metrics).to_csv(OUTPUT / 'example_metrics.csv', index=False)
pd.DataFrame(example_recommendations).to_csv(OUTPUT / 'example_recommendations.csv', index=False)
for name, frame in [('baseline', baseline), ('depth', depth_summary), ('threshold', threshold_summary),
                    ('pool', pool_summary), ('equal_length', equal_summary)]:
    frame.to_csv(OUTPUT / f'{name}_summary.csv', index=False)


def git_output(*args):
    try:
        return subprocess.check_output(['git', *args], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


source_files = [ROOT / f'src/ourney_travelsync/{f}.py' for f in ['evaluation', 'common', 'features', 'places']]
source_files += sorted((ROOT / 'src/ourney_travelsync/recommenders').glob('*.py'))
manifest = {
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(), 'python_executable': sys.executable,
    'numpy_version': np.__version__, 'pandas_version': pd.__version__,
    'dataset': OFFICIAL_FILENAME, 'dataset_sha256': dataset_hash, 'poi_count': len(places.place_ids),
    'git_revision': git_output('rev-parse', 'HEAD'), 'git_status': git_output('status', '--porcelain'),
    'source_sha256': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files},
    'seeds': list(SEEDS), 'profiles': {'food': food_profile, 'nature': nature_profile}, 'scenarios': scenarios,
    'baseline': {'candidate_top_n': CANDIDATE_TOP_N, 'top_k': TOP_K, 'awm_threshold': AWM_THRESHOLD, 'gfar_depth': GFAR_DEPTH},
    'gfar_depths': list(DEPTHS), 'awm_thresholds': list(THRESHOLDS),
    'pool_comparison': {'top_k': TOP_K, 'awm_threshold': AWM_THRESHOLD, 'gfar_depth': 10},
    'equal_length': {'pool': 'full', 'top_k': 5, 'awm_threshold': AWM_THRESHOLD, 'gfar_depth': 10},
    'notebook_source_sha256': hashlib.sha256(json.dumps(
        [(c['cell_type'], c['source']) for c in json.loads((ROOT / 'notebooks/compare_recommenders.ipynb').read_text(encoding='utf-8'))['cells']],
        ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest(),
    'run_count': len(results_frame), 'selected_poi_records': len(selected_frame),
    'utility_averaging': 'Nonempty runs only; empty-run utilities are undefined.',
}
(OUTPUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print(f'Saved {len(results_frame)} runs and per-POI evidence to {OUTPUT}')

Saved 450 runs and per-POI evidence to d:\Backup\Documents\Inha '23\8th semester\Capstone Design in CSE\TravelSync\Ourney_TravelSync\data\generated\evaluation\compare_recommenders


<a id="limitations"></a>
## Interpretation and next steps

The results expose a tradeoff between predicted group utility, member protection, and list completeness. Evaluate candidate expansion before interpreting an empty AWM list as group incompatibility. GFAR depth 10 warrants more study in these fixtures; it is not a validated production default. Keep production settings unchanged pending broader evaluation.

**Limits of this evidence:**

- Three extreme four-member groups use only two distinct profiles. Seeds perturb ties, not the user population.
- Evaluation uses the same category model that produces scores. There are no independent relevance labels or real-user satisfaction measurements.
- Parameters and the five-item comparison were explored on the same fixtures; there is no held-out validation set.
- Runtime, geography, travel feasibility, budget, opening hours, accessibility, and routing were not evaluated.
- Equivalence of this GFAR implementation to the authors' implementation has not been established.

**Next experiments:** add varied profiles and group sizes, hold settings fixed on new groups, compare candidate policies that retain compromise places, and define an explicit response when fewer than K acceptable POIs exist. Report member utilities and list lengths together.